# AIC26 — NVILA Field-Based Captioning (end-to-end)

Single self-contained notebook. Captions every AIC keyframe into a **structured visual-field JSON**
(+ optional caption & image embeddings) to power V-KIS / T-KIS retrieval. Runs on **Kaggle GPU T4 x2**.

**Before running:**
1. Add data: attach `aic-dataset-2025-s1` and `aic-dataset-2025-s2` (Add Input).
2. Settings -> Accelerator -> **GPU T4 x2**. Internet -> **On**.
3. Run cells top to bottom. Edit **CONFIG** first (set `LIMIT` small for a test run).

**Output:** `/kaggle/working/captions.jsonl` (one record per keyframe). To resume after the
~9h session limit, save it as a Kaggle Dataset output and re-attach next session — the run
auto-skips already-done keyframes.

**V-KIS reality:** the query clip is only *projected* on a screen (no digital file) with on-screen
text removed, so the operator DESCRIBES what they saw. Matching uses text-independent `VKIS_FIELDS`
+ caption/image embeddings (index cells). `text_on_screen` is captured but EXCLUDED from V-KIS.

**Sharding across 5 accounts:** set `VIDEO_PREFIX_RANGE` (e.g. `("L21","L30")`) per account.


In [ ]:
# --- installs (NVILA needs transformers 4.46 + scaling_on_scales for Dynamic-S2) ---
!pip -q install "transformers==4.46" accelerate opencv-python torchvision einops pillow
!pip -q install git+https://github.com/bfshi/scaling_on_scales.git
# only needed for the optional INDEX / EMBED cells at the very end:
!pip -q install -U sentence-transformers


In [ ]:
import glob

# ===================== CONFIG (edit me) =====================
ROOTS = sorted(glob.glob("/kaggle/input/*"))          # auto: every attached dataset
MODEL_ID = "Efficient-Large-Model/NVILA-Lite-2B-hf-preview"   # fits 1xT4. 8B needs a 24GB+ GPU or the dispatch_model trick (see engine cell)
OUT_DIR = "/kaggle/working"
CAPTIONS = OUT_DIR + "/captions.jsonl"

DO_A2 = True                 # also caption a short clip for action_motion (more time)
A2_RADIUS = 2                # neighbor frames each side for the A2 clip
DEDUP = True                 # skip near-duplicate keyframes (copy fields from representative)
DEDUP_THRESHOLD = 6          # dHash hamming distance
VIDEO_PREFIX_RANGE = None    # e.g. ("L21", "L30") to shard this account; None = all videos
LIMIT = 50                   # cap keyframes for a first test; set None for the full run
MAX_NEW_TOKENS = 256
# ============================================================
print("roots:", ROOTS)
print("model:", MODEL_ID)


In [ ]:
import csv, json, re, glob
from pathlib import Path
import numpy as np
from PIL import Image

HARD_FILTER_VOCAB = {
    "scene_type": ["indoor", "outdoor", "studio", "street", "stage", "mixed", "unknown"],
    "location_type": ["urban", "rural", "coastal", "highway", "field", "room", "water", "sky", "unknown"],
    "time_of_day": ["day", "night", "dusk", "unknown"],
    "weather": ["sunny", "cloudy", "rainy", "na", "unknown"],
    "camera_shot": ["close-up", "medium", "wide", "aerial", "unknown"],
    "num_people": ["0", "1", "2", "few", "crowd"],
}
ACTION_TAGS = ["speaking", "walking", "running", "vehicle-moving", "fire-spreading", "gathering", "static"]
EVENT_TAGS = ["anchor-desk", "interview", "press-conference", "speech-podium", "outdoor-report",
              "accident", "fire", "flood", "storm", "traffic", "protest", "ceremony", "sports",
              "weather-map", "chart-graphic", "building-exterior", "agriculture", "military",
              "crowd", "meeting", "disaster", "market"]
SOFT_FIELDS = ["event_type", "people_roles", "people_description", "key_objects",
               "dominant_colors", "action_motion", "distinctive_detail", "summary", "text_on_screen"]

# Fields usable for V-KIS matching: text-INDEPENDENT visual cues only. The shown V-KIS clip has
# on-screen text removed AND is only projected (the operator describes from observation, there is no
# digital clip). So text_on_screen is EXCLUDED from V-KIS (kept for T-KIS / QA + general indexing).
VKIS_FIELDS = ["scene_type", "location_type", "time_of_day", "weather", "camera_shot", "num_people",
               "event_type", "people_roles", "people_description", "key_objects",
               "dominant_colors", "action_motion", "distinctive_detail", "summary"]

A1_DEFAULTS = {
    "scene_type": "unknown", "location_type": "unknown", "time_of_day": "unknown", "weather": "na",
    "camera_shot": "unknown", "num_people": "0", "event_type": [], "people_roles": [],
    "people_description": "", "key_objects": [], "dominant_colors": [],
    "text_on_screen": "", "distinctive_detail": "", "summary": "", "uncertain_fields": [],
}


def make_keyframe_id(video_id, n):
    return f"{video_id}#{int(n):03d}"


def normalize_hard_fields(values):
    """Coerce any out-of-vocab Tier-1 value to 'unknown' ('0' for num_people); report coerced fields."""
    fixed, uncertain = dict(values), []
    for field, vocab in HARD_FILTER_VOCAB.items():
        v = fixed.get(field)
        if field == "num_people":
            v = str(v)
            fixed[field] = v
        if v not in vocab:
            fixed[field] = "0" if field == "num_people" else "unknown"
            uncertain.append(field)
    return fixed, uncertain


def scan_roots(roots, verbose=True):
    """Fast shallow scan of the AIC layout. Avoids recursive ** globs, which crawl the
    entire ~190GB tree (hundreds of thousands of files) and stall for many minutes on
    Kaggle's network filesystem. Videos_* is not scanned (A2 uses neighbor keyframes)."""
    jpg_dirs, csvs = {}, {}
    kf_patterns = ["Keyframes_*/keyframes/*", "*/Keyframes_*/keyframes/*"]
    csv_patterns = ["map-keyframes*/*/*.csv", "map-keyframes*/*.csv",
                    "*/map-keyframes*/*/*.csv", "*/map-keyframes*/*.csv"]
    for root in roots:
        root = Path(root)
        for pat in kf_patterns:
            for kf in root.glob(pat):
                if kf.is_dir():
                    jpg_dirs[kf.name] = kf
        for pat in csv_patterns:
            for c in root.glob(pat):
                csvs.setdefault(c.stem, c)
        if verbose:
            print(f"  {root}: {len(jpg_dirs)} videos, {len(csvs)} csv")
    assets = {vid: {"video_id": vid, "jpg_dir": jdir,
                    "csv_path": csvs.get(vid), "video_path": None}
              for vid, jdir in jpg_dirs.items()}
    return assets


def parse_map_keyframes(csv_path, video_id, jpg_dir):
    refs = []
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            n = int(row["n"])
            refs.append({
                "keyframe_id": make_keyframe_id(video_id, n),
                "video_id": video_id, "n": n,
                "frame_idx": int(row["frame_idx"]),
                "pts_time": float(row["pts_time"]),
                "fps": float(row["fps"]),
                "jpg_path": str(Path(jpg_dir) / f"{n:03d}.jpg"),
            })
    refs.sort(key=lambda r: r["n"])
    return refs


def dhash_to_int(path, size=8):
    """Difference hash -> 64-bit int (scipy-free); for near-dup grouping of repeated frames."""
    img = Image.open(path).convert("L").resize((size + 1, size), Image.BILINEAR)
    px = np.asarray(img, dtype=np.int16)
    diff = px[:, 1:] > px[:, :-1]
    value = 0
    for b in diff.flatten():
        value = (value << 1) | int(b)
    return value


def hamming(a, b):
    return (a ^ b).bit_count()


def group_near_dups(items, threshold):
    """items=[(keyframe_id, hash_int)] in n-order. Returns keyframe_id -> representative id."""
    reps, mapping = [], {}
    for kid, h in items:
        match = next((rid for rid, rh in reps if hamming(h, rh) <= threshold), None)
        if match is None:
            reps.append((kid, h))
            mapping[kid] = kid
        else:
            mapping[kid] = match
    return mapping


def build_a1_prompt():
    lines = "\n".join(f"- {f}: one of [{', '.join(v)}]" for f, v in HARD_FILTER_VOCAB.items())
    return (
        "You are labeling a single news video keyframe. Output ONLY one JSON object, no extra text.\n"
        "Hard-filter fields (pick exactly one allowed value each):\n" + lines + "\n"
        "Then these fields:\n"
        "- event_type: list of tags from [" + ", ".join(EVENT_TAGS) + "]\n"
        "- people_roles: list (anchor, reporter, official, police, soldier, civilian, athlete, ...)\n"
        "- people_description: short English phrase (clothing colors, gender/age cues)\n"
        "- key_objects: list of salient visible objects\n"
        "- dominant_colors: 2-3 color words\n"
        "- distinctive_detail: ONE short phrase for a VISUAL cue (shape/object/layout). Do NOT quote on-screen text.\n"
        "- summary: ONE English sentence describing what is VISUALLY happening (scene, people, action). "
        "Describe only what is seen; do NOT read or quote any on-screen text.\n"
        "- text_on_screen: verbatim on-screen text if any (may be Vietnamese); '' if none.\n"
        "- uncertain_fields: list of field names you are unsure about.\n"
        "Use English everywhere except text_on_screen. If a value does not apply use 'unknown' (or 'na' for weather).\n"
        'Example: {"scene_type":"studio","location_type":"room","time_of_day":"day","weather":"na",'
        '"camera_shot":"medium","num_people":"1","event_type":["anchor-desk"],"people_roles":["anchor"],'
        '"people_description":"woman in red blazer","key_objects":["desk","microphone"],'
        '"dominant_colors":["blue","white"],"distinctive_detail":"curved news desk",'
        '"summary":"A female anchor in red sits at a studio desk presenting to camera.",'
        '"text_on_screen":"BAN TIN","uncertain_fields":[]}'
    )


def build_a2_prompt():
    return (
        "These frames are consecutive moments around one keyframe of a news video. "
        "Identify the motion/action. Output ONLY JSON: {\"action_motion\": [tags]} where each tag is one of ["
        + ", ".join(ACTION_TAGS) + "]. Use 'static' if nothing moves."
    )


def extract_json(raw):
    raw = (raw or "").strip()
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        raise ValueError("no JSON object in model output: " + repr(raw[:80]))
    return json.loads(m.group(0))


def parse_a1(raw):
    try:
        data = {**A1_DEFAULTS, **extract_json(raw)}
    except Exception:
        data = dict(A1_DEFAULTS)
        data["uncertain_fields"] = list(HARD_FILTER_VOCAB)
    fixed, coerced = normalize_hard_fields(data)
    out = {k: fixed.get(k, A1_DEFAULTS[k]) for k in A1_DEFAULTS}
    out["uncertain_fields"] = sorted(set(list(out.get("uncertain_fields", []) or []) + coerced))
    for k in ["event_type", "people_roles", "key_objects", "dominant_colors"]:
        if not isinstance(out[k], list):
            out[k] = [out[k]] if out[k] else []
    out["event_type"] = [t for t in out["event_type"] if t in EVENT_TAGS]
    for k in ["people_description", "text_on_screen", "distinctive_detail", "summary"]:
        out[k] = str(out.get(k, "") or "")
    return out


def parse_a2(raw):
    try:
        data = extract_json(raw)
        return [t for t in data.get("action_motion", []) if t in ACTION_TAGS]
    except Exception:
        return []


def merge_record(ref, a1, a2_tags, did_a2, dedup_group, model_version):
    rec = dict(ref)
    rec.update(a1)
    rec["action_motion"] = list(a2_tags) if a2_tags else []
    rec["dedup_group"] = dedup_group
    rec["model_version"] = model_version
    rec["passes"] = {"a1": True, "a2": bool(did_a2)}
    rec["schema_version"] = "2.0"
    return rec


def vkis_soft_text(record):
    """Concatenate text-INDEPENDENT soft fields for embedding (V-KIS text<->caption).
    Excludes text_on_screen on purpose (the V-KIS clip has text removed)."""
    fields = ["summary", "event_type", "people_roles", "people_description",
              "key_objects", "action_motion", "dominant_colors", "distinctive_detail"]
    parts = []
    for f in fields:
        v = record.get(f, "")
        parts.append(" ".join(v) if isinstance(v, list) else str(v))
    return " | ".join(p for p in parts if p)


def append_jsonl(rec, path):
    with open(path, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()


def load_done_ids(path):
    ids = set()
    if Path(path).exists():
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    ids.add(json.loads(line)["keyframe_id"])
                except Exception:
                    pass
    return ids


In [ ]:
# --- load NVILA on a single GPU ---
import torch, PIL.Image
from transformers import AutoModel

print("loading", MODEL_ID, "...")
# NVILA casts its submodules to fp16 inside __init__, which is INCOMPATIBLE with accelerate
# device_map offload ("RuntimeError: can't move a model offloaded to cpu"). So load WITHOUT
# device_map and put the whole model on one GPU. (Lite-2B fp16 ~4GB fits one T4.)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).eval()
model = model.to("cuda")
# To run the 8B on 2xT4 instead: keep the line above WITHOUT .to("cuda"), then spread it:
#   from accelerate import dispatch_model, infer_auto_device_map
#   dmap = infer_auto_device_map(model, max_memory={0: "14GiB", 1: "14GiB"})
#   model = dispatch_model(model, device_map=dmap)
try:
    model.generation_config.max_new_tokens = MAX_NEW_TOKENS
except Exception as e:
    print("note: could not set max_new_tokens:", e)


def _gen(media_and_prompt):
    with torch.no_grad():
        return model.generate_content(media_and_prompt)   # NVILA HF API; returns a string


def caption_image(jpg_path):
    img = PIL.Image.open(jpg_path).convert("RGB")
    return parse_a1(_gen([img, build_a1_prompt()]))


def caption_clip(frame_paths):
    imgs = [PIL.Image.open(p).convert("RGB") for p in frame_paths]
    return parse_a2(_gen(imgs + [build_a2_prompt()]))


# --- smoke test on the first available keyframe ---
_assets = scan_roots(ROOTS)
_first = next((a for a in _assets.values() if a["csv_path"]), None)
assert _first is not None, "No video with a map-keyframes CSV found. Check attached datasets/ROOTS."
_refs = parse_map_keyframes(_first["csv_path"], _first["video_id"], _first["jpg_dir"])
print("smoke A1 ->", caption_image(_refs[0]["jpg_path"]))
if DO_A2:
    print("smoke A2 ->", caption_clip([r["jpg_path"] for r in _refs[:3]]))


In [ ]:
# --- main end-to-end captioning loop (resumable) ---
import time

assets = scan_roots(ROOTS)
done = load_done_ids(CAPTIONS)
print(f"{len(assets)} videos found; {len(done)} keyframes already captioned")


def _in_range(vid):
    if VIDEO_PREFIX_RANGE is None:
        return True
    lo, hi = VIDEO_PREFIX_RANGE
    return lo <= vid.split("_")[0] <= hi


count, t0, stop = 0, time.time(), False
for vid in sorted(assets):
    if stop:
        break
    a = assets[vid]
    if not _in_range(vid) or not a["csv_path"]:
        continue
    refs = parse_map_keyframes(a["csv_path"], vid, a["jpg_dir"])
    if DEDUP:
        items = [(r["keyframe_id"], dhash_to_int(r["jpg_path"])) for r in refs]
        groups = group_near_dups(items, DEDUP_THRESHOLD)
    else:
        groups = {r["keyframe_id"]: r["keyframe_id"] for r in refs}

    for idx, ref in enumerate(refs):
        if ref["keyframe_id"] in done:
            continue
        try:
            a1 = caption_image(ref["jpg_path"])
            if DO_A2:
                lo, hi = max(0, idx - A2_RADIUS), min(len(refs), idx + A2_RADIUS + 1)
                a2 = caption_clip([refs[j]["jpg_path"] for j in range(lo, hi)])
            else:
                a2 = []
            rec = merge_record(ref, a1, a2, DO_A2, groups[ref["keyframe_id"]], MODEL_ID)
            append_jsonl(rec, CAPTIONS)
            done.add(ref["keyframe_id"])
            count += 1
        except Exception as e:
            print("ERR", ref["keyframe_id"], repr(e)[:140])
        if count and count % 20 == 0:
            print(f"  {count} new | {count / (time.time() - t0):.2f} kf/s")
        if LIMIT and count >= LIMIT:
            stop = True
            break

print(f"DONE: wrote {count} new records -> {CAPTIONS} in {time.time() - t0:.0f}s")


## Optional: build the V-KIS index (run AFTER captioning)

`facets.sqlite` = Tier-1 enums for hard filtering. `cap_vectors.npy` = caption (text-independent)
embeddings for **text->caption** soft matching. Set `RUN_INDEX = True` and run the next cell.


In [ ]:
RUN_INDEX = False   # flip to True once captions.jsonl is populated

if RUN_INDEX:
    import sqlite3, json, numpy as np
    rows = [json.loads(l) for l in open(CAPTIONS) if l.strip()]

    # facet index (hard-filter enums)
    cols = ["keyframe_id", "video_id"] + list(HARD_FILTER_VOCAB)
    db = OUT_DIR + "/facets.sqlite"
    con = sqlite3.connect(db)
    con.execute("DROP TABLE IF EXISTS facets")
    con.execute("CREATE TABLE facets (%s)" % ", ".join(c + " TEXT" for c in cols))
    for r in rows:
        con.execute("INSERT INTO facets VALUES (%s)" % ",".join(["?"] * len(cols)),
                    [str(r.get(c, "")) for c in cols])
    for c in HARD_FILTER_VOCAB:
        con.execute(f"CREATE INDEX idx_{c} ON facets ({c})")
    con.commit(); con.close()
    print("facets ->", db, "(", len(rows), "rows )")

    # caption embeddings (text-independent visual fields only) for V-KIS text<->caption
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer("BAAI/bge-m3")
    ids = [r["keyframe_id"] for r in rows]
    vecs = np.asarray(enc.encode([vkis_soft_text(r) for r in rows], normalize_embeddings=True), dtype="float32")
    np.save(OUT_DIR + "/cap_vectors.npy", vecs)
    json.dump(ids, open(OUT_DIR + "/cap_ids.json", "w"))
    print("caption embeddings ->", vecs.shape)


## Optional: keyframe IMAGE embeddings — the main V-KIS lever

The V-KIS clip is only projected (no digital file), so image->image is impossible. Instead the
operator DESCRIBES what they saw and we match that text against keyframe **image** embeddings via
SigLIP's shared text<->image space (**text->image search**). Set `RUN_IMG_EMBED = True`.

At query time: `q = sig.get_text_features(**proc(text=[operator_description], return_tensors="pt").to("cuda"))`,
L2-normalize, then cosine vs `img_vectors.npy`.


In [ ]:
RUN_IMG_EMBED = False   # keyframe image embeddings (SigLIP) for V-KIS text->image search
SIGLIP_ID = "google/siglip-so400m-patch14-384"   # strong; faster/smaller: "google/siglip-base-patch16-224"

if RUN_IMG_EMBED:
    import torch, json, time, numpy as np
    from transformers import AutoModel, AutoProcessor
    from PIL import Image

    sig = AutoModel.from_pretrained(SIGLIP_ID).to("cuda").eval()
    proc = AutoProcessor.from_pretrained(SIGLIP_ID)

    def _in_range(vid):
        if VIDEO_PREFIX_RANGE is None:
            return True
        lo, hi = VIDEO_PREFIX_RANGE
        return lo <= vid.split("_")[0] <= hi

    assets = scan_roots(ROOTS)
    refs = []
    for vid in sorted(assets):
        a = assets[vid]
        if a["csv_path"] and _in_range(vid):
            refs += parse_map_keyframes(a["csv_path"], vid, a["jpg_dir"])

    ids, chunks, B, t0 = [], [], 32, time.time()
    for i in range(0, len(refs), B):
        batch = refs[i:i + B]
        imgs = [Image.open(r["jpg_path"]).convert("RGB") for r in batch]
        inp = proc(images=imgs, return_tensors="pt").to("cuda")
        with torch.no_grad():
            v = sig.get_image_features(**inp)
        v = torch.nn.functional.normalize(v, dim=-1).cpu().numpy().astype("float32")
        chunks.append(v)
        ids += [r["keyframe_id"] for r in batch]
        if (i // B) % 20 == 0:
            print(f"  embedded {len(ids)} | {len(ids) / (time.time() - t0 + 1e-9):.1f} img/s")

    np.save(OUT_DIR + "/img_vectors.npy", np.concatenate(chunks))
    json.dump(ids, open(OUT_DIR + "/img_ids.json", "w"))
    print("image embeddings ->", len(ids))


## Notes

- **Resume:** rerun the loop cell anytime — already-captioned `keyframe_id`s are skipped.
  Persist `/kaggle/working/captions.jsonl` as a **Kaggle Dataset output** between sessions.
- **Full run:** set `LIMIT = None`. **Shard** across your 5 accounts with `VIDEO_PREFIX_RANGE`.
- **V-KIS matching (downstream):** the clip is projected (no file) with text removed, so the operator
  DESCRIBES it. Match via (1) field-fill on `VKIS_FIELDS` (text-independent), (2) text->caption on the
  `summary`/soft embeddings (`cap_vectors.npy`), (3) text->image on the SigLIP keyframe embeddings
  (`img_vectors.npy`) using SigLIP's text tower at query time. `text_on_screen` is EXCLUDED from V-KIS
  (kept for T-KIS / QA).
- **`summary` / `distinctive_detail`** describe VISUAL content only (no quoting on-screen text).
- **Higher quality model:** run the 8B via the `dispatch_model` snippet in the engine cell (2xT4),
  or NVILA-15B on a bigger GPU.
- If `generate_content` errors on your transformers build, the model card also documents an
  `AutoProcessor` + `model.generate(...)` path as an alternative.
